### Supabase DB 및 Gemini 임베딩을 이용한 RAG 데이터 적재 (v1.2.3)
- 로직은 PolicyRec_v1_2_1.ipynb 참고
- 대상 CSV: main_v1_1_6.csv
- 필요한 키 : GOOGLE_API_KEY, SUPABASE_URL, SUPABASE_SERVICE_KEY

In [4]:
import os
import google.generativeai as genai
import pandas as pd
import json
import time
from dotenv import load_dotenv
from supabase import create_client, Client

load_dotenv(override=True)
genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_SERVICE_KEY = os.getenv("SUPABASE_SERVICE_KEY")

# 클라이언트 초기화
supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)

print("환경 설정 및 Supabase 클라이언트 초기화 완료")

c:\Users\user\miniconda3\envs\ai_agent\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\user\AppData\Local\Temp\ipykernel_22104\1525037603.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


환경 설정 및 Supabase 클라이언트 초기화 완료


### 1. 데이터 로드 및 전처리

In [2]:
# 1. 데이터 로드
csv_path = "data/csv/main/main_v1_1_6.csv"
df = pd.read_csv(csv_path)

# 2. 임베딩용 텍스트 구성 (기존 로직 유지)
def combine_features(row):
    return f"제목: {row['title']}\n카테고리: {row['category']}\n지역: {row['region']}\n대상: {row['target_group']}\n요약: {row['summary']}"

df['combined_text'] = df.apply(combine_features, axis=1)

print(f"총 {len(df)}개의 데이터를 로드했습니다.")
display(df.head(2))

총 554개의 데이터를 로드했습니다.


,source,source_id,title,summary,s_category,provider,region,target_group,target_age,target_age_min,...,application_method,detail_url,_scope,_scope_reason,norm_title,norm_provider,norm_period,category,subcategory,combined_text
0,biz,PBLN_000000000121348,2026년 한-체코ㆍ한-중국 에너지국제공동R&D 신규지원 대상과제 공고,"<p>2026년 한-체코,한-중국 에너지국제공동연구사업의 신규지원 대상 연구개발과제...",기술,한국에너지기술평가원,기후에너지환경부,중소기업,NaN,NaN,...,온라인 접수(범부처통합연구지원시스템),https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,main,primary,2026년한체코ㆍ한중국에너지국제공동rd신규지원대상과제공고,한국에너지기술평가원,사업별 상이~,기술,공동기술개발,제목: 2026년 한-체코ㆍ한-중국 에너지국제공동R&D 신규지원 대상과제 공고\n카...
1,biz,PBLN_000000000121347,[경남] 2026년 소재부품 성장 잠재기업 육성사업 수요기업 모집 공고,<p>한국생산기술연구원 첨단하이브리드생산기술센터에서는 동부경남지역 소재부품분야 관련...,기술,한국생산기술연구원,경상남도,중소기업,NaN,NaN,...,이메일 접수 (hsb85@kitech.re.kr),https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,main,primary,경남2026년소재부품성장잠재기업육성사업수요기업모집공고,한국생산기술연구원,2026-04-23~2026-05-07,기술,기술사업화/이전/지도,제목: [경남] 2026년 소재부품 성장 잠재기업 육성사업 수요기업 모집 공고\n카...


### 2. Gemini 임베딩 생성 (gemini-embedding-001 모델, 출력차원을 768차원으로 고정)

In [ ]:
# 1. 임베딩 생성 함수
def get_embedding(text):
    model_name = "models/gemini-embedding-001"
    result = genai.embed_content(
        model=model_name,
        content=text,
        task_type="retrieval_document",
        output_dimensionality=768
    )
    return result['embedding']

def split_text(text, max_length=1500):
    if len(text) <= max_length:
        return [text]
    chunks = []
    for i in range(0, len(text), max_length):
        chunks.append(text[i:i + max_length])
    return chunks

# 2. 경로 및 저장 설정
embedding_dir = "data/embedding"
embedding_file = os.path.join(embedding_dir, "embedded_announcements_v1_2_3.json")
os.makedirs(embedding_dir, exist_ok=True)

insert_data = []

# 3. 기존 임베딩 파일 확인
if os.path.exists(embedding_file):
    print(f"기존 임베딩 파일을 불러옵니다: {embedding_file}")
    with open(embedding_file, 'r', encoding='utf-8') as f:
        insert_data = json.load(f)
    print(f"성공: {len(insert_data)}개의 데이터를 로드했습니다.")
else:
    print("새로운 임베딩을 생성합니다. (API 호출 발생)")
    
    # 원본 파일 경로 매핑 사전 정의
    source_file_map = {
        'biz': r'data\raw\biz\bizinfo_page1_size10_20260422_115241.json',
        'kst': r'data\raw\kst\kstartup_page1_size10_20260422_115241.json',
        'youth': r'data\raw\youth\youthcenter_page1_size10_20260422_115241.json'
    }

    for idx, row in df.iterrows():
        full_text = row['combined_text']
        chunks = split_text(full_text, max_length=1500)
        
        for chunk in chunks:
            try:
                embedding = get_embedding(chunk)
                
                # CSV의 모든 컬럼을 가져옴
                data = row.to_dict()
                
                # 모든 컬럼 전처리 (NaN 및 '확인필요' NULL 처리)
                for key, val in data.items():
                    if val == '확인필요' or pd.isna(val):
                        data[key] = None
                
                # source_file 매핑 로직
                source_val = data.get('source')
                if source_val in source_file_map:
                    data['source_file'] = source_file_map[source_val]
                
                # 필수 데이터 및 임베딩 추가
                data["content"] = chunk
                data["embedding"] = embedding
                
                # 불필요한 임시 컬럼 제외
                if 'combined_text' in data:
                    del data['combined_text']
                if 'additional_conditions' in data:     #  추가
                    del data['additional_conditions']   #  추가
                
                insert_data.append(data)
                time.sleep(0.5) # rate limit 방지
            except Exception as e:
                print(f"임베딩 생성 오류 (Index {idx}): {e}")
    
    # 파일 저장
    with open(embedding_file, 'w', encoding='utf-8') as f:
        json.dump(insert_data, f, ensure_ascii=False, indent=2)
    print(f"임베딩 데이터 저장이 완료되었습니다: {embedding_file}")

print(f"최종 준비 완료된 청크 개수: {len(insert_data)}")


기존 임베딩 파일을 불러옵니다: data/embedding\embedded_announcements_v1_2_3.json
성공: 554개의 데이터를 로드했습니다.
최종 준비 완료된 청크 개수: 554


### 3.임베딩한 json 파일과 db의 컬럼 매핑후 Supabase `announcements` 테이블에 데이터 적재 

>1. 나이 데이터 타입 자동 형변환 (Type Casting)

CSV 처리 중 빈 값(NaN)이 섞여 자동으로 실수형(float)으로 변환된 연령 컬럼(target_age_min, target_age_max)을 식별합니다.
DB의 정수형(integer) 스키마와 충돌(22P02 에러)하지 않도록, 39.0 같은 값들을 int 타입(39)으로 강제 변환합니다.
변환할 수 없거나 빈 값인 경우 안전하게 None(DB의 NULL)으로 처리합니다.

>2. 네트워크 안정성 확보 및 재시도 (Retry) 로직

한 번에 대량의 데이터를 밀어넣을 때 발생하는 부하를 줄이기 위해 batch_size = 1 (1건씩) 단위로 순차 적재합니다.
일시적인 인터넷 끊김이나 SSL 통신 오류(SSLV3_ALERT_BAD_RECORD_MAC)가 발생하더라도 뻗지 않도록, 예외 처리(Try-Except) 및 최대 5회 자동 재시도 로직이 구현되어 있습니다.
오류 발생 시 Supabase 클라이언트를 다시 초기화하고 일정 시간 대기(time.sleep)한 후 적재를 이어갑니다.

In [9]:
import os
import json
import time
import requests
from dotenv import load_dotenv

load_dotenv(override=True)
url = os.getenv("SUPABASE_URL")
key = os.getenv("SUPABASE_SERVICE_KEY")

# 1. Supabase에서 테이블 스키마(컬럼 정보) 동적 조회
headers = {
    "apikey": key,
    "Authorization": f"Bearer {key}"
}
res = requests.get(f"{url}/rest/v1/", headers=headers)
spec = res.json()
table_columns = set(spec['definitions']['announcements']['properties'].keys())

# 2. 로컬 임베딩 JSON 데이터 읽어오기
embedding_file = "data/embedding/embedded_announcements_v1_2_3.json"
print(f"[{embedding_file}] 읽는 중...")
with open(embedding_file, 'r', encoding='utf-8') as f:
    insert_data = json.load(f)

# 3. 컬럼 비교 세팅
if insert_data:
    json_columns = set(insert_data[0].keys())
else:
    json_columns = set()

mapped_columns = json_columns & table_columns

# 4. 컬럼명 매핑 및 필터링 (타입 변환 포함 -> 실수형을 정수형(int)으로 형변환 처리 + 날짜형변환)
print("DB 스키마에 맞게 컬럼명을 변경하고, 날짜와 나이 데이터를 변환합니다...")

rename_map = {
    'title': 'raw_title',
    'apply_start': 'apply_start_dt',
    'apply_end': 'apply_end_dt'
}

def parse_date(date_val):
    """ '20260424' -> '2026-04-24' 로 변환, '사업별 상이' 등은 None 처리 """
    if not date_val:
        return None
    date_str = str(date_val).strip()
    
    # 8자리 숫자면 연-월-일 로 포맷 변경
    if len(date_str) == 8 and date_str.isdigit():
        return f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:]}"
    # 이미 'YYYY-MM-DD' 형식이면 그대로 통과
    elif len(date_str) >= 10 and "-" in date_str:
        return date_str
    
    # "사업별 상이", "상시접수" 등의 텍스트는 날짜타입에 못 넣으므로 None 처리
    return None

filtered_data = []
for data in insert_data:
    filtered_row = {}
    for k, v in data.items():
        # 1) DB 스키마 이름으로 매핑
        mapped_key = rename_map.get(k, k)
        
        if mapped_key in table_columns:
            # 2-1) 나이 소수점 처리 로직
            if mapped_key in ['target_age_min', 'target_age_max']:
                if v is not None:
                    try:
                        filtered_row[mapped_key] = int(float(v))
                    except (ValueError, TypeError):
                        filtered_row[mapped_key] = None
                else:
                    filtered_row[mapped_key] = None
                    
            # 2-2) 날짜(timestamp) 텍스트/포맷 예외 처리 로직 추가
            elif mapped_key in ['apply_start_dt', 'apply_end_dt']:
                filtered_row[mapped_key] = parse_date(v)
                
            # 2-3) 나머지 일반 텍스트 데이터
            else:
                filtered_row[mapped_key] = v
                
    filtered_data.append(filtered_row)


# 5. DB 적재 진행
print(f"총 {len(filtered_data)}개의 데이터 적재를 시작합니다...")
batch_size = 1
supabase = get_supabase_client() # 상단 셀에서 정의한 함수 사용

for i in range(0, len(filtered_data), batch_size):
    batch = filtered_data[i:i + batch_size]
    success = False
    
    for retry in range(5):
        try:
            supabase.table("announcements").insert(batch).execute()
            if (i + 1) % 10 == 0 or (i + 1) == len(filtered_data):
                print(f"[{i+1}/{len(filtered_data)}] 적재 중...")
            success = True
            break 
        except Exception as e:
            print(f"⚠️ [{i+1}번 데이터] 오류: {e}")
            supabase = get_supabase_client()
            time.sleep((retry + 1) * 2)
    
    if not success:
        print(f"❌ {i+1}번 데이터 적재 실패.")

print("\n적재 완료되었습니다!")


[data/embedding/embedded_announcements_v1_2_3.json] 읽는 중...
DB 스키마에 맞게 컬럼명을 변경하고, 날짜와 나이 데이터를 변환합니다...
총 554개의 데이터 적재를 시작합니다...
[10/554] 적재 중...
[20/554] 적재 중...
[30/554] 적재 중...
[40/554] 적재 중...
[50/554] 적재 중...
[60/554] 적재 중...
[70/554] 적재 중...
[80/554] 적재 중...
[90/554] 적재 중...
[100/554] 적재 중...
[110/554] 적재 중...
[120/554] 적재 중...
[130/554] 적재 중...
[140/554] 적재 중...
[150/554] 적재 중...
[160/554] 적재 중...
⚠️ [169번 데이터] 오류: Server disconnected
[170/554] 적재 중...
[180/554] 적재 중...
[190/554] 적재 중...
[200/554] 적재 중...
[210/554] 적재 중...
[220/554] 적재 중...
[230/554] 적재 중...
[240/554] 적재 중...
[250/554] 적재 중...
[260/554] 적재 중...
[270/554] 적재 중...
[280/554] 적재 중...
[290/554] 적재 중...
[300/554] 적재 중...
[310/554] 적재 중...
[320/554] 적재 중...
[330/554] 적재 중...
[340/554] 적재 중...
[350/554] 적재 중...
⚠️ [356번 데이터] 오류: [SSL: SSLV3_ALERT_BAD_RECORD_MAC] ssl/tls alert bad record mac (_ssl.c:2590)
[360/554] 적재 중...
[370/554] 적재 중...
[380/554] 적재 중...
[390/554] 적재 중...
[400/554] 적재 중...
[410/554] 적재 중...
[420/554] 

### 5. 적재된 데이터 5건 조회

In [10]:
# Supabase에서 5건을 조회하여 출력합니다.
try:
    response = supabase.table("announcements").select("*").limit(5).execute()
    recent_data = response.data
    
    if recent_data:
        print("\n[ 적재된 데이터 5건 조회 ]")
        df_recent = pd.DataFrame(recent_data)
        
        # 임베딩 등 너무 긴 컬럼은 제외하고 출력
        cols_to_show = [col for col in df_recent.columns if col not in ['embedding', 'content']]
        display(df_recent[cols_to_show])
    else:
        print("조회된 데이터가 없습니다.")
except Exception as e:
    print(f"데이터 조회 중 오류 발생: {e}")


[ 적재된 데이터 5건 조회 ]


,id,source,source_id,raw_title,summary,provider,norm_title,norm_provider,norm_period,s_category,...,target_age_min,target_age_max,apply_start_dt,apply_end_dt,target_group,detail_url,_scope,_scope_reason,created_dt,updated_dt
0,755,biz,PBLN_000000000121348,2026년 한-체코ㆍ한-중국 에너지국제공동R&D 신규지원 대상과제 공고,"<p>2026년 한-체코,한-중국 에너지국제공동연구사업의 신규지원 대상 연구개발과제...",한국에너지기술평가원,2026년한체코ㆍ한중국에너지국제공동rd신규지원대상과제공고,한국에너지기술평가원,사업별 상이~,기술,...,NaN,NaN,NaN,NaN,중소기업,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,main,primary,2026-04-28T07:21:53.215892+00:00,2026-04-28T07:21:53.215892+00:00
1,756,biz,PBLN_000000000121347,[경남] 2026년 소재부품 성장 잠재기업 육성사업 수요기업 모집 공고,<p>한국생산기술연구원 첨단하이브리드생산기술센터에서는 동부경남지역 소재부품분야 관련...,한국생산기술연구원,경남2026년소재부품성장잠재기업육성사업수요기업모집공고,한국생산기술연구원,2026-04-23~2026-05-07,기술,...,NaN,NaN,2026-04-23T00:00:00+00:00,2026-05-07T00:00:00+00:00,중소기업,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,main,primary,2026-04-28T07:21:53.340103+00:00,2026-04-28T07:21:53.340103+00:00
2,757,biz,PBLN_000000000121346,[경기] 이천시 2026년 도ㆍ공예기업 맞춤형 온라인 마케팅 지원사업 참여기업 모집 공고,<p>이천시와 경기테크노파크에서는 관내 도ㆍ공예기업 우수제품의 온라인 매출 증대와 ...,경기테크노파크,경기이천시2026년도ㆍ공예기업맞춤형온라인마케팅지원사업참여기업모집공고,경기테크노파크,2026-04-23~2026-05-22,판로/수출,...,NaN,NaN,2026-04-23T00:00:00+00:00,2026-05-22T00:00:00+00:00,중소기업,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,main,primary,2026-04-28T07:21:53.407226+00:00,2026-04-28T07:21:53.407226+00:00
3,758,biz,PBLN_000000000121345,[경기] 2026년 가상융합 사업화지원 (TRACK 2 : 도쿄 XR페어 참가 지원...,<p>경기콘텐츠진흥원은 아래와 같이 ‘2026년 가상융합 기업 사업화지원’에 참여할...,경기콘텐츠진흥원,경기2026년가상융합사업화지원track2도쿄xr페어참가지원연장공고,경기콘텐츠진흥원,2026-04-24~2026-04-30,경영,...,NaN,NaN,2026-04-24T00:00:00+00:00,2026-04-30T00:00:00+00:00,중소기업,https://www.bizinfo.go.kr/sii/siia/selectSIIA2...,main,primary,2026-04-28T07:21:53.463756+00:00,2026-04-28T07:21:53.463756+00:00
4,1014,kst,177319,울산 바이오헬스기업 글로벌 진출 지원사업,울산지역 바이오‧디지털 헬스케어 기업의 글로벌 시장 진출을 지원하기 위해 BIO U...,교육기관,울산바이오헬스기업글로벌진출지원사업,교육기관,20260422~20260518,판로/수출,...,20.0,39.0,2026-04-22T00:00:00+00:00,2026-05-18T00:00:00+00:00,일반기업,https://www.k-startup.go.kr/web/contents/bizpb...,main,primary,2026-04-28T07:22:08.716601+00:00,2026-04-28T07:22:08.716601+00:00
